# 8.2 Peer-Review — '정확도 99.8%'는 좋은 모델인가, 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter08_2_peer_review_metrics.ipynb)

책 본문: [8.2 Peer-Review](https://smhanlab.com/book-ml/kor/ml1/chapter08/2.html)

이 노트북은 8.2절의 '가상 발표'를 코드로 재현합니다. 신용카드 이상거래 데이터(정상 99.8% / 사기 0.2%)에서
(1) **정확도의 함정**이 실제로 어떤 숫자로 나타나는지, (2) 정확도가 거의 안 움직이는 동안 재현율은
0에서 0.99로 크게 바뀌는 모습, (3) 책의 가설(로지스틱회귀 vs GBDT)을 재현해 정확도 차이가 '우연'인 것,
(4) 이 숫자들로 **반박 가능한 리뷰 질문**을 어떻게 만들어야 하는지 보여줍니다.


In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt

IMG = "/home/smhan/book-ml/kor/src/images"   # (Colab에서는 /tmp로 바꾸면 됨)


## 1. 데이터: 0.2%가 사기인 1차원 장난감 데이터

발표 요약의 설정을 그대로 씁니다 — 거래 100,000건, 그중 200건(0.2%)이 사기. 한 특징 $x$만 놓고,
정상 거래는 $\mathcal{N}(0,1)$, 사기 거래는 $\mathcal{N}(4,1)$이라고 둡니다. $x$가 클수록 사기에
가깝습니다. 시드(42)를 고정해 누가 실행해도 같은 숫자가 나옵니다.


In [2]:
np.random.seed(42)
N = 100_000
n_fraud = int(0.002 * N)      # 200
n_norm  = N - n_fraud          # 99_800

x_norm  = np.random.randn(n_norm)        # 정상: N(0,1)
x_fraud = 4.0 + np.random.randn(n_fraud) # 사기:   N(4,1)
x = np.concatenate([x_norm, x_fraud])
y = np.concatenate([np.zeros(n_norm), np.ones(n_fraud)])

base_rate = y.mean()
print(f"총 {N:,}건, 사기 {n_fraud}건, 정상 {n_norm:,}건")
print(f"베이스레이트(양성률) = {base_rate:.5f}")


총 100,000건, 사기 200건, 정상 99,800건
베이스레이트(양성률) = 0.00200


## 2. 정확도의 함정 — '쓸모있는 모델'의 정확도는 '무조건 정상'보다 낮다

임계값 $t$를 넘으면 사기로 판정하는 가장 단순한 분류자입니다. $t=2.5$(두 분포 사이에서 사기 쪽에
가까운 위치)를 쓰면 아래처럼 **사기의 90%를 잡지만**, 정확도는 99.37%에 불과합니다. 그런데 **아무
모델 없이 전부 '정상'으로 찍기만 하면** 정확도는 99.8%입니다 — 실제로는 사기를 단 1건도 못 잡는데도요.


In [3]:
def report(x, y, t):
    pred = (x > t).astype(int)
    TP = int(((pred==1)&(y==1)).sum()); FP = int(((pred==1)&(y==0)).sum())
    FN = int(((pred==0)&(y==1)).sum()); TN = int(((pred==0)&(y==0)).sum())
    acc  = (TP+TN)/len(y)
    prec = TP/(TP+FP) if (TP+FP) else 0.0
    rec  = TP/(TP+FN) if (TP+FN) else 0.0
    f1   = 2*prec*rec/(prec+rec) if (prec+rec) else 0.0
    return dict(t=t, TP=TP, FP=FP, FN=FN, TN=TN, acc=acc, prec=prec, rec=rec, f1=f1)

useful = report(x, y, t=2.5)
for k, v in useful.items(): print(f"  {k:4s}: {v}")

all_normal_acc = 1 - base_rate   # 전부 '정상'으로 찍는다면
print(f"")
print(f"쓸모있는 모델(t=2.5):  accuracy={useful['acc']:.5f}  recall={useful['rec']:.4f}  precision={useful['prec']:.4f}")
print(f"무조건 '정상':        accuracy={all_normal_acc:.5f}  recall=0.0000  precision=0")
print(f"-> 정확도는 '무조건 정상'이 더 높다({all_normal_acc:.5f} > {useful['acc']:.5f}), 사기를 잡는 건 쓸모있는 쪽이 {useful['rec']:.1%}뿐.")


  t   : 2.5
  TP  : 180
  FP  : 611
  FN  : 20
  TN  : 99189
  acc : 0.99369
  prec: 0.22756005056890014
  rec : 0.9
  f1  : 0.36326942482341074

쓸모있는 모델(t=2.5):  accuracy=0.99369  recall=0.9000  precision=0.2276
무조건 '정상':        accuracy=0.99800  recall=0.0000  precision=0
-> 정확도는 '무조건 정상'이 더 높다(0.99800 > 0.99369), 사기를 잡는 건 쓸모있는 쪽이 90.0%뿐.


In [4]:
ts = np.linspace(1.0, 5.0, 200)
accs, recs = [], []
for tt in ts:
    r = report(x, y, tt)
    accs.append(r['acc']); recs.append(r['rec'])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
ax = axes[0]
ax.plot(ts, accs, color="#1d4ed8", lw=2)
ax.axhline(all_normal_acc, color="#dc2626", ls="--", lw=1.2, label=f"Always 'normal' = {all_normal_acc:.4f}")
ax.axvline(2.5, color="#495057", ls=":", lw=1)
ax.text(2.52, 0.985, f"t=2.5 (useful model)\nacc={useful['acc']:.4f}", fontsize=8, color="#111")
ax.set_ylim(0.97, 1.0)
ax.set_xlabel("threshold t"); ax.set_ylabel("Accuracy")
ax.set_title("Accuracy: barely moves")
ax.legend(fontsize=8, loc="lower left")

ax = axes[1]
ax.plot(ts, recs, color="#0f5132", lw=2)
ax.axvline(2.5, color="#495057", ls=":", lw=1)
ax.axvline(3.5, color="#c0392b", ls=":", lw=1)
ax.text(3.55, 0.5, "t=3.5: rec=0.71", fontsize=8, color="#c0392b")
ax.text(2.55, 0.95, "t=2.5: rec=0.90", fontsize=8, color="#0f5132")
ax.set_xlabel("threshold t"); ax.set_ylabel("Recall")
ax.set_title("Recall: changes dramatically from 0 to 1")
fig.tight_layout()
fig.savefig(IMG + "/ch08_2_accuracy_trap.svg")
plt.show()


## 3. 더 정확할수록 사기를 더 놓친다 — 임계값 실험

같은 분류자에서 임계값만 $3.5 \to 2.5 \to 2.0$으로 낮춰가면(사기를 더 잡으려는 쪽) 정확도는
낮아지고 재현율은 올라갑니다. **정확도로 줄을 세우면 $t=3.5$가 '최고', $t=2.0$이 '최하'**인데,
사기를 실제로 잡는 능력(재현율)으로는 순서가 **완전히 반대**입니다. 이 '순서 역전'이 정확도의
함정의 핵심입니다.


In [5]:
rows = [3.5, 2.5, 2.0]
print(f"{'t':>4}  {'accuracy':>9}  {'precision':>10}  {'recall':>7}  {'F1':>7}")
for t in rows:
    r = report(x, y, t)
    print(f"{t:4.1f}  {r['acc']:9.5f}  {r['prec']:10.4f}  {r['rec']:7.4f}  {r['f1']:7.4f}")
print("")
print("정확도 순위: t=3.5 > t=2.5 > t=2.0   (t=3.5가 '최고')")
print("재현율 순위: t=2.0 > t=2.5 > t=3.5   (t=2.0이 '최고')   <- 순서가 반대로!")


   t   accuracy   precision   recall       F1
 3.5    0.99919      0.8606   0.7100   0.7781
 2.5    0.99369      0.2276   0.9000   0.3633
 2.0    0.97721      0.0783   0.9650   0.1448

정확도 순위: t=3.5 > t=2.5 > t=2.0   (t=3.5가 '최고')
재현율 순위: t=2.0 > t=2.5 > t=3.5   (t=2.0이 '최고')   <- 순서가 반대로!


## 4. 책의 가설 재현: 로지스틱회귀 vs GBDT

발표 요약을 그대로 재현합니다 — 같은 불균형 데이터에 로지스틱회귀와 GBDT를 적용하고 정확도·PR-AUC를
비교합니다. 기대되는 관찰: **두 모델의 정확도는 99.8% 부근에서 사실상 동점**(발표에서 근거로 든
99.7% vs 99.8% 차이는 이 잡음보다 작거나 같은 크기)이고, 의미를 가진 지표 **PR-AUC**에서야 차이가
드러납니다.


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import average_precision_score, accuracy_score
from sklearn.model_selection import train_test_split

rng = np.random.RandomState(0)
n2 = 100_000; n2_pos = int(0.002 * n2)
neg = rng.randn(n2 - n2_pos, 2); pos = rng.randn(n2_pos, 2) + np.array([2.5, 2.5])
X = np.vstack([neg, pos]); Y = np.concatenate([np.zeros(n2-n2_pos), np.ones(n2_pos)])
Xtr, Xte, Ytr, Yte = train_test_split(X, Y, test_size=0.5, random_state=0, stratify=Y)

lr = LogisticRegression(max_iter=1000).fit(Xtr, Ytr)
gb = GradientBoostingClassifier(n_estimators=50, max_depth=3, learning_rate=0.1).fit(Xtr, Ytr)

print(f"테스트셋 베이스레이트 = {Yte.mean():.5f}")
print("")
for name, mdl in [("로지스틱회귀", lr), ("GBDT", gb)]:
    proba = mdl.predict_proba(Xte)[:,1]
    accv = accuracy_score(Yte, (proba>0.5).astype(int))
    prv  = average_precision_score(Yte, proba)
    print(f"{name:8s}  accuracy@0.5={accv:.5f}   PR-AUC={prv:.4f}")


테스트셋 베이스레이트 = 0.00200

로지스틱회귀    accuracy@0.5=0.99864   PR-AUC=0.6357
GBDT      accuracy@0.5=0.99800   PR-AUC=0.0705


## 5. 이 숫자로 '반박 가능한 질문' 만들기

리뷰의 핵심은 막연한 "더 잘 설명해주세요"가 아니라, **이 학기에 배운 개념 하나를 근거로** 무엇이
부족한지 짚는 것입니다. 위에서 나온 숫자를 대입해 실제로 던질 질문을 만들어봅니다.


In [7]:
q1 = ("정확도 " + f"{useful['acc']:.4f}" + "는 '무조건 정상'(" + f"{all_normal_acc:.4f}"
      + ")보다 낮은데도 모델이 쓸모있다고 주장하십니다. "
      + "발표의 두 모델(로지스틱회귀 vs GBDT)은 정확도 99.8% 부근에서 사실상 동점인데, "
      + "이 0.1%p 차이는 Chapter 6.3에서 배운 검증셋 표본의 잡음 크기보다 큰 근거가 될 수 있나요? "
      + "Chapter 2.3의 재현율/PR-AUC를 보여주지 않는 한, 두 모델이 실제 사기를 얼마나 더 잡는지 구분이 안 됩니다.")
print(q1)


정확도 0.9937는 '무조건 정상'(0.9980)보다 낮은데도 모델이 쓸모있다고 주장하십니다. 발표의 두 모델(로지스틱회귀 vs GBDT)은 정확도 99.8% 부근에서 사실상 동점인데, 이 0.1%p 차이는 Chapter 6.3에서 배운 검증셋 표본의 잡음 크기보다 큰 근거가 될 수 있나요? Chapter 2.3의 재현율/PR-AUC를 보여주지 않는 한, 두 모델이 실제 사기를 얼마나 더 잡는지 구분이 안 됩니다.


## 요약

- **정확도의 함정**: 베이스레이트가 99.8%인 데이터에서 '무조건 정상'만 찍어도 정확도 99.8%다.
  쓸모있는 모델(재현율 90%)의 정확도(99.37%)는 그보다 **낮다**.
- **순서 역전**: 임계값을 올리면 정확도는 오르지만 재현율은 0.90에서 0.71로 떨어진다. 정확도 순위와
  '사기를 잡는' 순위가 **반대**다.
- **동점의 정확도**: 로지스틱회귀 vs GBDT의 정확도 차이는 잡음 수준. 의미를 가진 지표는 **PR-AUC/재현율**.
- **반박 가능한 질문** = 이 개념(Ch02.3 정확도의 함정, Ch06.3 검증의 잡음)을 근거로, 발표팀이
  실제로 결과를 다시 들여다보아야만 답할 수 있는 질문.
